# Background to HOPS

For the ExoClock project, HOPS (Holly Observatory Pipeline Software) is used to process light curves from astronomical observations. To prepare a file for use by HOPS, follow these steps:

1. Image Format
HOPS works best with FITS files. Ensure your images are in this format.
If you have a sequence of images, they should be in a single folder for easy batch processing.

2. Image Naming
The filenames should be consistent and preferably include timestamps.
Example: exoplanet_YYYYMMDD_HHMMSS.fits

3. Metadata and Headers
Ensure your FITS files have the required headers:

DATE-OBS: Start time of exposure in UTC (YYYY-MM-DDTHH:MM:SS.SSS)
EXPTIME: Exposure time in seconds
FILTER: Filter used (e.g., R, I, V)
TELESCOP: Telescope name
INSTRUME: Camera name
AIRMASS: Airmass at observation time
RA and DEC: Coordinates of the target

If any metadata is missing, you might need to update the FITS headers.

4. Calibration Frames
Bias, dark, and flat frames should be available for calibration.
Ensure you have master bias, dark, and flat frames ready.

5. Time Synchronization
Time accuracy is crucial for exoplanet transit timing.
Ensure that the timestamps in DATE-OBS are synchronized with UTC.
If using a local PC, ensure the system clock is accurate (consider using NTP synchronization).

6. Data Submission Format
If submitting to ExoClock, follow their naming and file structure guidelines.

You might need to upload a ZIP archive containing:
FITS images (science frames)
Calibration frames (if applicable)
Observation log (CSV or TXT file with metadata)

# libraries and functions

In [ ]:
import os
import pandas as pd
import zipfile
from astropy.io import fits
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
def extract_selected_files(zip_path, extract_to, condition_func):
    i = 0

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for file_name in zip_ref.namelist():
            i += 1
            print(".", end="")  # Print dot without newline
            if i % 10 == 0:
                print(f' {i} Files extracted')  # Print a newline every 10 iterations

            if condition_func(file_name):  # Apply the condition

                new_name = os.path.basename(file_name)  # Remove directory structure
                new_path = os.path.join(extract_to, new_name)
                
                with zip_ref.open(file_name) as source, open(new_path, "wb") as target:
                    target.write(source.read())  # Write the file contents manually                



In [3]:
def check_fits_headers(directory):
    required_headers = ["DATE-OBS", "EXPTIME", "FILTER", "TELESCOP", "INSTRUME", "AIRMASS", "RA", "DEC"]
    
    fits_files = [f for f in os.listdir(directory) if f.endswith(".fits")]
    if not fits_files:
        print("No FITS files found in the directory.")
        return
    
    for file in fits_files:
        file_path = os.path.join(directory, file)
        with fits.open(file_path) as hdul:
            hdr = hdul[0].header
            print(f"\nChecking: {file}")
            
            missing_headers = [key for key in required_headers if key not in hdr]
            if missing_headers:
                print(f"  Missing headers: {', '.join(missing_headers)}")
            else:
                print("  All required headers are present.")
            
            # Additional check for DATE-OBS format
            if "DATE-OBS" in hdr:
                try:
                    from datetime import datetime
                    datetime.strptime(hdr["DATE-OBS"], "%Y-%m-%dT%H:%M:%S")
                except ValueError:
                    print("  Warning: DATE-OBS format may be incorrect (expected YYYY-MM-DDTHH:MM:SS).")



In [ ]:
def extract_fits_from_fz(fz_file, output_dir=None):
    """Extracts FITS file from a .fz compressed file and saves it."""
    
    # Ensure the output directory exists
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Open the .fz file
    with fits.open(fz_file) as hdul:
        # Construct output filename (remove .fz extension)
        output_file = os.path.splitext(fz_file)[0] # + ".fits"
        if output_dir:
            output_file = os.path.join(output_dir, os.path.basename(output_file))
        
        # Save the extracted FITS file
        hdul.writeto(output_file, overwrite=True)
        print(f"Extracted: {output_file}")

## Extract from download .zip

In [ ]:
# Example usage
current_dir = os.getcwd()  # Get the current working directory
zip_file_path = os.path.join(current_dir, "data", "y.zip")

try:
    zip_file_path = os.path.join(current_dir, "HATS-38b", "lco_data-20250327-422.zip")
    extract_directory = os.path.join(current_dir, "HATS-38b", "data")
    condition = lambda name: 'e91' in name

    extract_selected_files(zip_file_path, extract_directory, condition)
except FileNotFoundError:
    print(f"Error: The file {zip_file_path} was not found.")
except zipfile.BadZipFile:
    print(f"Error: The file {zip_file_path} is not a zip file.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

In [25]:
def get_fits_headers(full_file_name, verbose = False):

    try:
        list_hdu = []
        list_tag = []
        list_value = []
        list_description = []
        if verbose:
            print(f"Reading: {full_file_name}")

        with fits.open(full_file_name) as hdul:
            for i, hdu in enumerate(hdul):
                x = repr(hdu.header).split('\n')
                for j in x:
                    list_hdu.append(i)
                    j = j.replace('=',',')
                    j = j.replace('/',',')
                    y = j.split(',')
                    y = [item.strip() for item in y if item.strip()]
                    if len(y) == 2:
                        y.append('')

                    list_tag.append(y[0])
                    list_value.append(y[1])
                    list_description.append(y[2])
        
        return(list_hdu, list_tag, list_value, list_description)
    
    except Exception as e:
        print(f"Error: {e}")
        return None



# HOPS file prep

In [27]:
from astropy.io import fits

current_dir = os.getcwd()  # Get the current working directory
file_path = os.path.join(current_dir, "HATS-38b", "lco_data-20250407-211")

fits_files = [f for f in os.listdir(file_path) if f.endswith(".fits.fz")]

try:
    for fileno, file_name in enumerate(fits_files):
        # Create a DataFrame from the lists
        list_hdu, list_tag, list_value, list_description = get_fits_headers(os.path.join(file_path, file_name))

        if list_hdu is None:
            print(f"Failed to read {file_name}. Skipping.")
            continue

        if fileno == 0:
            df = pd.DataFrame({
                'HDU': list_hdu,
                'Description': list_description,
                'Tag': list_tag,
                f'Value-{fileno}': list_value
            })
        else:
            df = df.copy()    
            df[f'Value-{fileno}'] = list_value
            
except Exception as e:
        print(f"Error reading FITS file: {e}")

In [ ]:
try:
    df.to_csv(os.path.join(file_path, "headers.csv"), index=False)
except Exception as e:
    print(f"Error saving DataFrame to CSV: {e}")

In [99]:
# extract the values for the plots
value_columns = [f'Value-{i}' for i in range(211)]

# x axis texts -------------------------------------------------------
xtext = '[UTC] Start date and time of the observati'
try:
    df_plot = df[df['Description'] == xtext].reset_index(drop=True)
    xtext = df_plot.iloc[0][value_columns].values.astype(str).tolist()
    xtext = [x[12:20] for x in xtext]
except Exception as e:
    print(f"Error filtering DataFrame: {e}")

# y axis values ------------------------------------------------------
flt = ['Effective mean airmass', 
       '[mbar] Atmospheric pressure',
       '[%] Current percentage humidity',
       '[deg C] External temperature']

try:
    df_plot = df[df['Description'].isin(flt)].reset_index(drop=True)

except Exception as e:
    print(f"Error processing DataFrame: {e}")


In [114]:
def get_normalised(list_of_values) :
    try:
        min_value = min(list_of_values)
        max_value = max(list_of_values)
        # return [(value - min_value) / (max_value - min_value) for value in list_of_values], min_value, max_value
        return [(value - min_value) / (max_value - min_value) for value in list_of_values]
    except Exception as e:
        print(f"Error normalizing values: {e}")
        return [], None, None

In [119]:
values = [10, 20, 30, 40]
normalized, min_v, max_v = get_normalised(values)
print(normalized)  # [0.0, 0.333..., 0.666..., 1.0]

Error normalizing values: 'numpy.float64' object is not callable
[]


In [112]:
fig = go.Figure()
x = list(range(1, 212))

try:
    for i in range(len(df_plot)):
        list_vals = df_plot.iloc[i][value_columns].values.astype(float).tolist()
        get_normalised(list_vals)
    #     y, min, max = get_normalised(list_vals)
    #     yname = f'{df_plot.iloc[i]['Description']} {min} - {max}'

    #     fig.add_trace(go.Scatter(x=x, y=y, name=yname,
    #                             text=xtext,           # This sets hover text per point
    #                             hoverinfo='text+y'))     # Show only the custom text and y value))

    # # Set the tick labels using xtexts
    # fig.update_layout(
    #     xaxis=dict(
    #         tickmode='array',
    #         tickvals=x[::10],       # Every 10th x value
    #         ticktext=xtext[::10],  # Matching every 10th label
    #         tickangle=45            # Rotate labels 45 degrees
    #     ),
    #     title='BANZAI Headers measures for HATS-38b, 2025-03-26',
    # )
    # fig.show()
except Exception as e:
    print(f"Error creating plot: {e}")

Error normalizing values: 'numpy.float64' object is not callable
Error normalizing values: 'numpy.float64' object is not callable
Error normalizing values: 'numpy.float64' object is not callable
Error normalizing values: 'numpy.float64' object is not callable
